# Диагностика модели: регрессия с серийной корреляцией

In [ ]:
import pandas as pd
import numpy as np

from statsmodels.formula.api import ols
from statsmodels.stats.api import durbin_watson, acorr_breusch_godfrey # DW & LM-тесты
from statsmodels.iolib.summary2 import summary_col # вывод подгонки

from scipy.stats import chi2 # chi2-распределение и критические значения

# Не показывать Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)

<div style="background-color:Bisque; color:DarkBlue; padding:30px;">

<i><b><span style="color: purple">Диагностика модели: тесты на серийную корреляцию</span></b></i><br>

Рассмотрим __линейную регрессию для временных рядов__ с серийной корреляцией для ошибки (внешнего шока)

$$
\begin{aligned}
y_t&=x_t^\top\beta+u_t \\ u_t&=\rho_1u_{t-1}+\cdots+\rho_pu_{t-p}+v_t
\end{aligned}
$$

где
* $y_t,x_t$ наблюдаемы
* $u_t$ — эндогенный внешний шок (__ненаблюдаем!__)
* $v_t$ — экзогенный внешний шок (__ненаблюдаем!__)
* $p$ — порядок серийной корреляции/автокорреляции

**<span style="color:purple">DW-тест:</span>** тестируем автокорреляцию __первого порядка__, т.е. для модели

$$
\begin{aligned}
y_t&=x_t^\top\beta+u_t \\ u_t&=\rho u_{t-1}+v_t
\end{aligned}
$$

тестируем

$$
\begin{aligned}
H_0:\rho&=0 & &vs & H_1:\rho&\ne0
\end{aligned}
$$

__Тестовая статистика__

$$
\begin{aligned}
DW&=\frac{\sum_{t=2}^n(e_t-e_{t-1})^2}{\sum_{t=1}^n e^2_t} & 0\leq DW&\leq4
\end{aligned}
$$

__Критические значения__: из таблицы критических значений [`tables/dw-critical-values.md`](../tables/dw-critical-values.md) находим два критических значения $0<d_L<d_U$ (зависят от $n,k,\alpha$)

__Вывод:__

|Значение $DW$ |Гипотеза $H_0$|Вывод |
|:--:|:--:|:--:|
|$0 \leq DW< d_L$|отвергается|есть положительная автокорреляция|
|$d_L \leq DW \leq d_U$ |?|неопределенность |
|$d_U < DW < 4-d_U$ |не отвергается |автокорреляция отсутствует |
|$4-d_U \leq DW \leq 4-d_L$|?|неопределенность |
|$4-d_L < DW \leq 4$ |отвергается|есть отрицательная автокорреляция|

__Вывод (приближённый):__
* Не отвергаем гипотезу при $DW\approx2$ (нет серийной корреляции)
* Отвергаем гипотезу если $DW$ "сильно отличается от 2" (есть серийная корреляция)

__Замечание:__

* используем метод [`statsmodels.stats.stattools.durbin_watson(resids)`](https://www.statsmodels.org/stable/generated/statsmodels.stats.stattools.durbin_watson.html#statsmodels.stats.stattools.durbin_watson)
* где `resids` — остатки модели
* критические значения берём из специальной таблицы

**<span style="color:purple">LM/BG-тест:</span>** тест на серийную корреляцию произвольного порядка

$$
H_0:\rho_1=\cdots=\rho_p=0
$$

Тест основан на вспомогательной (неинтерпретируемой!) регрессии для OLS-остатков

$$
\begin{aligned}
e_t\;&\text{на}\; e_{t-1},\ldots,e_{t-p}, x & &R^2_{aux}
\end{aligned}
$$

__Два подхода к тестированию__
* тестовая статистика $LM=nR^2_{aux}$, критическое значение $\chi^2_{cr}=\chi^2_{df=p}(\alpha)$ ($p$ — порядок серийной корреляции)
* F-тест на совместную значимость $e_{t-1},\ldots,e_{t-p}$

__Вывод:__ Тест указывает на серийную корреляцию (отвергаем $H_0$) при $LM>\chi^2_{cr}, F>F_{cr},P<\alpha$.

__Замечание:__

* используем метод [`statsmodels.stats.diagnostic.acorr_breusch_godfrey(res, nlags)`](https://www.statsmodels.org/stable/generated/statsmodels.stats.diagnostic.acorr_breusch_godfrey.html#statsmodels.stats.diagnostic.acorr_breusch_godfrey)
* где `res` — подогнанная модель
* `nlags` — порядок автокорреляции
* метод возвращает четыре числа: $LM$-статистику и её P-значение, $F$-статистику и её P-значение

</div>

<div style="background-color:Bisque; color:DarkBlue; padding:30px;">

<i><b><span style="color: purple">Эконометрика регрессии с серийной корреляцией</span></b></i><br>

Рассмотрим __линейную регрессию для временных рядов__ с серийной корреляцией для ошибки (внешнего шока)

$$
\begin{aligned}
y_t&=x_t^\top\beta+u_t \\ u_t&=\rho_1u_{t-1}+\cdots+\rho_pu_{t-p}+v_t
\end{aligned}
$$

где
* $y_t,x_t$ наблюдаемы
* $u_t$ — эндогенный внешний шок (__ненаблюдаем!__)
* $v_t$ — экзогенный внешний шок (__ненаблюдаем!__)
* $p$ — порядок серийной корреляции/автокорреляции

__Цель__:
* оценить коэффициенты $\beta$
* тестировать гипотезы о коэффициентах (t-/F-тесты)

**<span style="color:purple">Подгонка модели:</span>** обычный OLS

**<span style="color:purple">Тестирование гипотез:</span>** необходимо использовать робастную оценку ковариационной матрицы коэффициентов.
__Для регрессии с серийной корреляцией используем HAC-оценку__ (HAC = Heteroskedasticity Autocorrelation Consistent) или оценку Newey-West.

При подгонке модели используем следующие настройки `.fit(cov_type='HAC', cov_kwds={'maxlags': L}, use_t=True)`

В `cov_kwds` в виде словаря задаём настройки ковариационной матрицы
* `'maxlags': L` задаёт число лагов; берём $L=\lfloor 4(n/100)^{2/9}\rfloor$ (Schwert rule, скобки $\lfloor\cdot\rfloor$ означают целую часть числа)

Параметр `use_t=True` означает, что P-значения вычисляются по t-распределению

</div>

Для набора данных `Mishkin` рассмотрим регрессию

$$
\Delta(pai3)\;\text{на}\;\Delta(tb3),\Delta(\log(cpi))
$$

где
* $\Delta x_t = x_t-x_{t-1}$: первая разность или __приращение__ (период к периоду)
* $\Delta \log(x_t) = \log x_t-\log x_{t-1}$: лог-доходность или __процентное приращение__ (период к периоду)

In [ ]:
# импорт данных
df = pd.read_csv('../datasets/Mishkin.csv')

In [ ]:
# т.к. данные месячные, зададим периодический временной индекс с 1950-02 по 1990-12
# параметр freq задаёт периодичность
df.index = pd.period_range(start='1950-02', end='1990-12', freq='M')
df

## Спецификация и подгонка

__Замечание__: сначала подготовим набор данных, т.е. выполним необходимые преобразования данных и добавим новые переменные:
* дифференцируем `pai3, tb3` (новые переменные `diff_pai3, diff_tb3`)
* log-дифференцируем `cpi` (новая переменная `diff_log_cpi`)

In [ ]:
# добавим новые переменные в набор данных
df[['diff_pai3', 'diff_tb3']] = df[['pai3', 'tb3']].diff()
df['diff_log_cpi'] = np.log(df['cpi']).diff()
df

In [ ]:
# спецификация модели через формулу
mod = ols(formula='diff_pai3~1+diff_tb3+diff_log_cpi', data=df)
# подгонка модели с ковариационной матрицей по умолчанию (неробастной)
res = mod.fit()

## DW-тест

Тестируем модель на серийную корреляцию 1-го порядка

In [ ]:
# DW-статистика
durbin_watson(res.resid)

Число наблюдений `.nobs` и число регрессоров модели `.df_model` для нахождения критических значений

In [ ]:
# число наблюдений и число регрессоров
mod.nobs, mod.df_model

Критические значения для $n=490$, $k=2$ и уровня значимости 5% берём из таблицы [`tables/dw-critical-values.md`](../tables/dw-critical-values.md): $d_L=1.844$, $d_U=1.860$.

**ВЫВОД**: на уровне значимости 5% $DW = 1.621 < d_L = 1.844$, гипотеза $H_0$ отвергается: есть положительная автокорреляция первого порядка.

## LM/BG-тест

Тестируем модель на серийную корреляцию 2-го порядка

In [ ]:
# LM-тест на автокорреляцию 2-го порядка
acorr_order = 2
lm, lmpval, fval, fpval = acorr_breusch_godfrey(res, nlags=acorr_order)
lm, lmpval

### Критическое значение $\chi^2$-распределения

In [ ]:
# уровень значимости
sign_level = 0.05
# критическое значение распределения chi2
chi2.isf(q=sign_level, df=acorr_order)

**ВЫВОД**: на уровне значимости 5% отвергаем $H_0$ (т.к. $LM>\chi^2_{cr}$): тест указывает на серийную корреляцию.

## Робастный t-тест (HAC s.e.)

Используем HAC-оценку ковариационной матрицы

In [ ]:
# спецификация модели через формулу
mod = ols(formula='diff_pai3~1+diff_tb3+diff_log_cpi', data=df)
# подгонка модели с ковариационной матрицей по умолчанию (неробастной)
res_ols = mod.fit()
# число лагов для HAC-оценки
L = int(4*(mod.nobs/100)**(2/9))
# подгонка модели с робастной HAC-оценкой ковариационной матрицы
res_hac = mod.fit(cov_type='HAC', cov_kwds={'maxlags': L}, use_t=True)

### Результаты двух оцениваний в одной таблице

__Обратим внимание на коды значимости (количество звёздочек у коэффициентов)__

In [ ]:
# неробастные и робастные (HAC) стандартные ошибки
summary_col(results=[res_ols, res_hac], model_names=['OLS-CME', 'HAC-CME'], stars=True)